<a id="top"></a>

# Demo: an inline tool definition *is* an MCP tool spec

```yaml
title: "Demo: an inline tool definition is an MCP tool spec"
keywords: mcp, tool spec, inline tool, input_schema, inputSchema, translate, json schema, l05 design, teacher demo, offline
estimated duration: 12
```

> **Lesson:** L09. **Pure Python (`json` only) — no API key, no `mcp` package, fully runnable here.**
>
> **The point to land:** an MCP **tool spec** is the [L08](../L08/L0802_lecture.md) design surface — name, description, schema — serialized. The *only* routine difference from the [L07](../L07/L0702_lecture.md) inline definition is one renamed key (`input_schema` → `inputSchema`). The JSON Schema itself survives the packaging change byte-for-byte. You'll prove that with `assert`s instead of a live server.

## Contents

- [1. Setup](#1-setup)
- [2. Translate inline -> MCP tool spec](#2-translate-inline---mcp-tool-spec)
- [3. The JSON Schema survives byte-for-byte](#3-the-json-schema-survives-byte-for-byte)
- [4. Round-trip back to inline](#4-round-trip-back-to-inline)
- [5. Takeaways](#5-takeaways)

## 1. Setup

Just `json` from the standard library, plus the [L08](../L08/L0802_lecture.md) `book_meeting` tool as an **inline** Anthropic definition. This is the design we will re-package — we are not redesigning it (that was L08's job).

In [1]:
import json

# The L08 `book_meeting` tool, as an Anthropic INLINE tool definition.
# This is the L07/L08 shape: note the key is `input_schema` (snake_case).
# Nothing here is MCP-specific yet — it is a well-designed tool, full stop.
BOOK_MEETING_INLINE: dict[str, object] = {
    "name": "book_meeting",
    "description": (
        "Book a meeting on the user's calendar. Use this when the user asks to "
        "schedule, set up, or book a meeting with a named attendee at a given time."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "attendee": {
                "type": "string",
                "description": "Attendee email, RFC 5322, e.g. 'priya@example.com'.",
            },
            "start": {
                "type": "string",
                "description": "ISO 8601 start datetime, e.g. '2026-06-16T14:00'.",
            },
            "duration_minutes": {
                "type": "integer",
                "description": "Meeting length in minutes, integer between 15 and 240.",
            },
        },
        "required": ["attendee", "start"],
    },
}


print("inline definition ready for:", BOOK_MEETING_INLINE["name"])

inline definition ready for: book_meeting


[↑ Back to top](#top)

## 2. Translate inline -> MCP tool spec

An MCP tool spec uses the **same** fields with one rename: the schema key is `inputSchema` (camelCase), not `input_schema`. `name`, `description`, and the JSON Schema *contents* are untouched. The translation is a key rename, nothing more.

In [2]:
def inline_to_mcp_spec(inline: dict[str, object]) -> dict[str, object]:
    """Serialize an Anthropic inline tool definition to an MCP tool spec.

    The only routine difference is the schema key name:
        inline uses `input_schema`; an MCP tool spec uses `inputSchema`.
    name, description, and the JSON Schema contents are carried over unchanged.

    Example input:
        {"name": "t", "description": "d", "input_schema": {...}}
    Example output:
        {"name": "t", "description": "d", "inputSchema": {...}}"""
    return {
        "name": inline["name"],
        "description": inline["description"],
        "inputSchema": inline["input_schema"],
    }


mcp_spec = inline_to_mcp_spec(BOOK_MEETING_INLINE)
print(json.dumps(mcp_spec, indent=2))

{
  "name": "book_meeting",
  "description": "Book a meeting on the user's calendar. Use this when the user asks to schedule, set up, or book a meeting with a named attendee at a given time.",
  "inputSchema": {
    "type": "object",
    "properties": {
      "attendee": {
        "type": "string",
        "description": "Attendee email, RFC 5322, e.g. 'priya@example.com'."
      },
      "start": {
        "type": "string",
        "description": "ISO 8601 start datetime, e.g. '2026-06-16T14:00'."
      },
      "duration_minutes": {
        "type": "integer",
        "description": "Meeting length in minutes, integer between 15 and 240."
      }
    },
    "required": [
      "attendee",
      "start"
    ]
  }
}


[↑ Back to top](#top)

## 3. The JSON Schema survives byte-for-byte

The claim from the [lecture](L0902_lecture.md) (slide 2.2): the *design* is unchanged. We check it mechanically — the schema object on both sides must be **equal**, and the only differing top-level key is the schema's name.

In [3]:
# The schema contents are identical across the two packagings.
assert mcp_spec["inputSchema"] == BOOK_MEETING_INLINE["input_schema"]
# name and description carry over untouched.
assert mcp_spec["name"] == BOOK_MEETING_INLINE["name"]
assert mcp_spec["description"] == BOOK_MEETING_INLINE["description"]
# The ONLY structural difference is the schema key name.
assert "input_schema" in BOOK_MEETING_INLINE and "inputSchema" in mcp_spec
print("design surface preserved: name, description, and JSON Schema all identical")

design surface preserved: name, description, and JSON Schema all identical


[↑ Back to top](#top)

## 4. Round-trip back to inline

Portability cuts both ways: an MCP spec discovered from a server can be registered with an inline client by reversing the rename. Round-tripping inline → MCP → inline must return the original definition exactly — proving no design information is lost in the packaging.

In [4]:
def mcp_spec_to_inline(spec: dict[str, object]) -> dict[str, object]:
    """Reverse of inline_to_mcp_spec: rename `inputSchema` back to `input_schema`."""
    return {
        "name": spec["name"],
        "description": spec["description"],
        "input_schema": spec["inputSchema"],
    }


round_tripped = mcp_spec_to_inline(mcp_spec)
assert round_tripped == BOOK_MEETING_INLINE
print("round-trip inline -> MCP -> inline is lossless")

round-trip inline -> MCP -> inline is lossless


[↑ Back to top](#top)

## 5. Takeaways

- An MCP **tool spec** is the [L08](../L08/L0802_lecture.md) design surface (name + description + schema) serialized — there is **nothing new to design**.
- The only routine difference from an [L07](../L07/L0702_lecture.md) inline definition is `input_schema` → `inputSchema`; the JSON Schema is identical.
- The translation is **lossless both ways**, which is *why* the same tool is portable across clients — that portability is the whole value proposition (lecture section 5).
- Error shaping isn't visible here because it lives in the tool *result*, not the spec — the build walkthrough [L0906](L0906_lecture.ipynb) shows it surviving the boundary.
- Next: the [L0904 lab](L0904_lab_empty.ipynb) has you validate and translate a spec yourself; [L0906](L0906_lecture.ipynb) shows the server that *publishes* one.

[↑ Back to top](#top)